Для Fine Tuning мы будем использовать ту предобработку датасета, что была для промтинга. Задача Fine Tuning это побить бейзлайн, который был выбит на промтинге с лёгкой моделью qwen и базовым промтом. В ходе Fine Tuning мы не будем отпиливать голову модели, верхние слои или переобучать все веса. Последнее особенно долго и неудобно. Поэтому воспользуемся LoRA адаптером.

## Универсальный код для загрузки датасета

In [5]:
import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU доступен: True
Устройство: Tesla T4
Память: 15.6 GB


In [6]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d xuanhuynh233/ielts-dataset
!unzip ielts-dataset.zip

train_df = pd.read_csv('train_final.csv')
test_df  = pd.read_csv('test_final.csv')

print(train_df.head())
print(f"Размер: {len(train_df)} строк")

print("\n=== TEST ===")
print(test_df.head())
print(f"Размер: {len(test_df)} строк")

def clean_df(df, name):
    initial_len = len(df)
    df = df.dropna(subset=['essay', 'band'])
    df = df[pd.to_numeric(df['band'], errors='coerce').notna()]
    df['band'] = df['band'].astype(float)
    removed = initial_len - len(df)
    print(f"[{name}] Удалено {removed} строк → осталось {len(df)}")
    return df

train_df = clean_df(train_df, 'TRAIN')
test_df  = clean_df(test_df,  'TEST')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset URL: https://www.kaggle.com/datasets/xuanhuynh233/ielts-dataset
License(s): MIT
ielts-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  ielts-dataset.zip
replace test_final.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: test_final.csv          
replace train_final.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: train_final.csv         
                                      band  \
0  7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
1  5.0\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
2  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
3  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
4    4\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   

                                              prompt  \
0  Interviews form the basic criteria for most la...   
1  Interviews form the basic selecting criteria f...   
2  I

## Загружаем qwen

In [7]:
# !pip uninstall -y bitsandbytes
# !pip install bitsandbytes>=0.46.1

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Конфигурация 4-bit квантизации - чтобы быстро скачать и чтобы быстро работало (для пробы пера)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = 'right' # Добавляем, чтобы нормально модель нормально обучалась

print(f'Параметров: {model.num_parameters():,}')

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Параметров: 3,085,938,688


## Промт

Не смотря на то, что мы обучаем модель, она не знает что мы ей даём на входе. Для обучения всё равно необходимо, чтобы модель понимала задачу, которую будет решать. Но здесь в отличие от чистого промтинга, не нужно указывать примеры, так как модель и так будет на них обучаться. Будем использовать instruction + zero-shot.

In [9]:
# сразу все импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
import re
from tqdm import tqdm
import torch

Будем учитывать само задание, чтобы модель более корректно оценивала Task Response

In [10]:
# промт на вход модели
def prompt(essay, task_prompt):
    return f"""You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Task: {task_prompt}

Essay: {essay}

Provide your evaluation in this EXACT format:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]"""

In [11]:
# правильный ответ
def answer(row):
    return f"""TR_Band: {row['TR_Band']}
CC_Band: {row['CC_Band']}
LR_Band: {row['LR_Band']}
GRA_Band: {row['GRA_Band']}
Overall_Band: {row['Overall_Band']}
Explanation: {row['General_Feedback']}"""

Во время обучения модель до того, как увидеть настоящий ответ, предсказывает свой и потом считает лосс в сравнении с настоящим.

In [12]:
# преобразуем данные в нужный формат для модели
def df_to_messages(df):
    records = []
    for _, row in df.iterrows():
        records.append({
            'messages': [
                {'role': 'user', 'content': prompt(row['essay'], row['prompt'])},
                {'role': 'assistant', 'content': answer(row)},
            ]
        })
    return records

## LoRA

In [13]:
!pip install -q peft trl

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16, # ранг матриц обновления
    lora_alpha=32, # скорость обучения
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention
        "gate_proj", "up_proj", "down_proj", # MLP
    ],
    lora_dropout=0.05, # регуляризация
    bias="none",
    task_type="CAUSAL_LM", # тип задачи
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


https://huggingface.co/docs/peft/package_reference/lora

Обучаем 0,96% весов модели

## Токенизация датасета

In [14]:
# !pip install -q pyarrow --upgrade
# перезапустить сеанс

In [15]:
from datasets import Dataset

max_length = 1024

def tokenize_dataset(records):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for rec in records:
        # строим полный чат
        text = tokenizer.apply_chat_template(
            rec['messages'],
            tokenize=False,
            add_generation_prompt=False
        )

        # токенизация
        tokenized = tokenizer(
            text,
            max_length=max_length,
            padding='max_length',
            truncation=True,
        )

        input_ids = tokenized['input_ids']
        attention_mask = tokenized['attention_mask']

        assistant_start = text.find('<|im_start|>assistant')
        prompt_part = tokenizer(text[:assistant_start], add_special_tokens=False)['input_ids']
        prompt_len = len(prompt_part)

        labels = input_ids.copy()
        labels[:prompt_len] = [-100] * prompt_len # заменяем на -100 всё, кроме ответа

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)

    return Dataset.from_dict({
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    })

In [31]:
train_df = train_df.sample(1000, random_state=42) #Проверяем на мелком

In [32]:
train_records = df_to_messages(train_df)
test_records = df_to_messages(test_df)

Нужно посмотреть, сколько наблюдений не влезло в 1024 токена. Посмотрим на тренировочной выборке, так как она больше.

In [33]:
overflow_count = 0
for rec in train_records:
    text = tokenizer.apply_chat_template(
        rec['messages'], tokenize=False, add_generation_prompt=False
    )
    tokens = tokenizer(text, add_special_tokens=False)['input_ids']
    if len(tokens) > max_length:
        overflow_count += 1

print(f"Примеров длиннее 1024 токенов: {overflow_count} из {len(train_records)}")
print(f"Это {overflow_count / len(train_records) * 100:.2f}% датасета")

Примеров длиннее 1024 токенов: 8 из 1000
Это 0.80% датасета


Не влезает 0,5% датасета. Чтобы уменьшить риск ошибок при обучении их лучше выкинуть до этапа токенизации датасета.

In [34]:
def filter_long_records(records, max_length):
    filtered = []
    for rec in records:
        text = tokenizer.apply_chat_template(
            rec['messages'], tokenize=False, add_generation_prompt=False
        )
        tokens = tokenizer(text, add_special_tokens=False)['input_ids']
        if len(tokens) <= max_length:
            filtered.append(rec)
    print(f"Отфильтровано {len(records) - len(filtered)} примеров, осталось {len(filtered)}")
    return filtered

In [35]:
train_records = filter_long_records(train_records, max_length)
test_records  = filter_long_records(test_records, max_length)

Отфильтровано 8 примеров, осталось 992
Отфильтровано 1 примеров, осталось 468


In [36]:
train_dataset = tokenize_dataset(train_records)
test_dataset = tokenize_dataset(test_records)

train_dataset.set_format('torch')
test_dataset.set_format('torch')

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")

# проверяем что маскировка работает
sample_labels = train_dataset[0]['labels']
n_masked = (sample_labels == -100).sum().item()
n_real = (sample_labels != -100).sum().item()
print(f"Замаскировано {n_masked} токенов (промт), обучаемых {n_real} токенов (ответ)")

Train: 992, Test: 468
Замаскировано 705 токенов (промт), обучаемых 319 токенов (ответ)


## Обучение

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

OUTPUT_DIR = "./qwen-ielts-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

print("Начинаем обучение...")
trainer.train()

Начинаем обучение...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss


In [ ]:
model.save_pretrained(OUTPUT_DIR + '/lora_adapter')
tokenizer.save_pretrained(OUTPUT_DIR + '/lora_adapter')